In [1]:
import datetime
import logging
import re
from pathlib import Path

import torch
import trackio
from datasets import load_dataset
from huggingface_hub.errors import GatedRepoError
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

/usr/local/lib/python3.11/dist-packages/trl/generation/__init__.py:22: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.19.0. You have version 0.21.0 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():
/usr/local/lib/python3.11/dist-packages/trl/generation/vllm_client.py:40: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.19.0. You have version 0.21.0 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():
/usr/local/lib/python3.11/dist-packages/trl/generation/vllm_generation.py:42: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.19.0. You have version 0.21.0 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():


In [2]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("gradio_client").setLevel(logging.WARNING)

In [3]:
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.current_device()}")
    print(f"GPU name: {torch.cuda.get_device_name()}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU available. This notebook requires a GPU for efficient training.")
    print("In Colab: Runtime → Change runtime type → Hardware accelerator → GPU")

CUDA available: True
Number of GPUs: 1
Current GPU: 0
GPU name: NVIDIA A40
GPU memory: 47.7 GB


## Configuration

In [4]:
_local_model_dir = Path("gemma-3-1b-it")
model_name = str(_local_model_dir) 
max_seq_length = 1024
max_prompt_length = 256
lora_rank = 32

print(f"model_name = {model_name}")

model_name = gemma-3-1b-it


In [5]:
reasoning_start = "<reasoning>"
reasoning_end = "</reasoning>"
solution_start = "<answer>"
solution_end = "</answer>"

SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

system_prompt = SYSTEM_PROMPT

print(system_prompt)


Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>



## Dataset: GSM8K

In [6]:
def extract_hash_answer(text):
    """Extract numerical answer from GSM8K format (#### marker)."""
    if "####" not in text:
        return None
    return text.split("####")[1].strip()


def process_dataset_example(example):
    """Convert GSM8K example to conversation format for GRPO training."""
    question = example["question"]
    answer = extract_hash_answer(example["answer"])
    prompt = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    return {
        "prompt": prompt,
        "answer": answer,
    }


gsm8k_local_dir = Path("/workspace/gsm8k-grade-school-math-8k-dataset/gsm8k/main")


def load_gsm8k_dataset():
    print("Loading GSM8K mathematical reasoning dataset...")
    dataset = load_dataset(
        "parquet",
        data_files={"train": str(gsm8k_local_dir / "train-00000-of-00001.parquet")},
        split="train",
    )
    dataset = dataset.map(process_dataset_example)

    assert "prompt" in dataset.column_names
    assert "answer" in dataset.column_names
    assert len(dataset[0]["prompt"]) == 2
    assert dataset[0]["answer"] is not None

    print("Dataset loaded and processed.")
    print(f"Training examples: {len(dataset):,}")
    print(f"Sample row: {dataset[0]}")
    print("-------------------------")
    print(f"Sample question: {dataset[0]['prompt'][1]['content']}...")
    print(f"Sample answer: {dataset[0]['answer']}")
    return dataset

In [7]:
dataset = load_gsm8k_dataset()
dataset[0]

Loading GSM8K mathematical reasoning dataset...
Dataset loaded and processed.
Training examples: 7,473
Sample row: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': '72', 'prompt': [{'role': 'system', 'content': '\nRespond in the following format:\n<reasoning>\n...\n</reasoning>\n<answer>\n...\n</answer>\n'}, {'role': 'user', 'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'}]}
-------------------------
Sample question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?...
Sample answer: 72


{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': '72',
 'prompt': [{'role': 'system',
   'content': '\nRespond in the following format:\n<reasoning>\n...\n</reasoning>\n<answer>\n...\n</answer>\n'},
  {'role': 'user',
   'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'}]}

## Prompt length check

Sanity-check `max_prompt_length` against the actual tokenized prompt lengths for this dataset/tokenizer, since it's an assumed value, not a derived one. Note this trl version's `GRPOConfig` has no `max_prompt_length` field, so prompts are never truncated - this only affects the `max_completion_length` budget (`max_seq_length - max_prompt_length`).

In [8]:
_length_check_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

_prompt_token_lengths = sorted(
    len(
        _length_check_tokenizer(
            _length_check_tokenizer.apply_chat_template(
                example["prompt"], add_generation_prompt=True, tokenize=False
            )
        )["input_ids"]
    )
    for example in dataset
)

_n = len(_prompt_token_lengths)
_over_budget = sum(1 for length in _prompt_token_lengths if length > max_prompt_length)

print(f"Prompts checked: {_n:,}")
print(f"min={_prompt_token_lengths[0]}  "
      f"p50={_prompt_token_lengths[_n // 2]}  "
      f"p90={_prompt_token_lengths[int(_n * 0.9)]}  "
      f"p95={_prompt_token_lengths[int(_n * 0.95)]}  "
      f"p99={_prompt_token_lengths[int(_n * 0.99)]}  "
      f"max={_prompt_token_lengths[-1]}")
print(f"max_prompt_length = {max_prompt_length}")
print(f"Prompts exceeding max_prompt_length: {_over_budget} ({_over_budget / _n:.2%})")

Prompts checked: 7,473
min=51  p50=94  p90=127  p95=139  p99=166  max=252
max_prompt_length = 256
Prompts exceeding max_prompt_length: 0 (0.00%)


### Completion length budget

tokenize the *reference* GSM8K solutions wrapped in our target format to see how long a **correct** completion actually needs to be.

In [9]:
_reference_answers = load_dataset(
    "parquet",
    data_files={"train": str(gsm8k_local_dir / "train-00000-of-00001.parquet")},
    split="train",
)["answer"]

def _format_reference_completion(answer_field):
    _reasoning, _final = answer_field.split("####")
    _reasoning = re.sub(r"<<.*?>>", "", _reasoning).strip()  # drop GSM8K calculator annotations
    return f"{reasoning_start}\n{_reasoning}\n{reasoning_end}\n{solution_start}{_final.strip()}{solution_end}"

_completion_token_lengths = sorted(
    len(_length_check_tokenizer(_format_reference_completion(ans))["input_ids"])
    for ans in _reference_answers
)

_m = len(_completion_token_lengths)
def _cpct(p):
    return _completion_token_lengths[min(int(_m * p), _m - 1)]

print(f"Reference completions checked: {_m:,}")
print(f"min={_completion_token_lengths[0]}  "
      f"p50={_cpct(0.5)}  p90={_cpct(0.9)}  p95={_cpct(0.95)}  "
      f"p99={_cpct(0.99)}  max={_completion_token_lengths[-1]}")
print("Coverage by max_completion_length cap:")
for _cap in [256, 384, 512, 640, 768]:
    _covered = sum(1 for length in _completion_token_lengths if length <= _cap) / _m
    print(f"  cap={_cap:4d} -> covers {_covered:6.1%} of reference solutions")

Reference completions checked: 7,473
min=38  p50=98  p90=163  p95=188  p99=235  max=358
Coverage by max_completion_length cap:
  cap= 256 -> covers  99.5% of reference solutions
  cap= 384 -> covers 100.0% of reference solutions
  cap= 512 -> covers 100.0% of reference solutions
  cap= 640 -> covers 100.0% of reference solutions
  cap= 768 -> covers 100.0% of reference solutions


### Reward Functions

| Reward | Max |
|---|---:|
| `xmlcount_reward_func` | `0.5` |
| `soft_format_reward_func` | `0.5` |
| `strict_format_reward_func` | `0.5` |
| `int_reward_func` | `0.5` |
| `correctness_reward_func` | `2.0` |
| **Total** | **4.0** |

The reward system teaches in this order:
1. Learn the tags.
2. Put the answer in the right place.
3. Make the answer an integer.
4. Make the answer correct.
5. Avoid messy text after the final answer.


In [10]:
def extract_xml_answer(text: str) -> str:
    """Extract the text between <answer> and </answer>."""
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()


def completion_texts(completions):
    """Return raw assistant text from TRL completion objects."""
    return [completion[0]["content"] for completion in completions]


#used by Trackio metrics
match_format = re.compile(
    r"^\s*<reasoning>.*?</reasoning>\s*<answer>\s*(.*?)\s*</answer>\s*$",
    flags=re.DOTALL,
)

In [11]:
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """Reward exact final-answer correctness."""
    responses = completion_texts(completions)
    extracted_responses = [extract_xml_answer(response) for response in responses]
    return [2.0 if response == true_answer else 0.0 for response, true_answer in zip(extracted_responses, answer)]

In [12]:
def int_reward_func(completions, **kwargs) -> list[float]:
    """Reward answers that are made only of digits."""
    responses = completion_texts(completions)
    extracted_responses = [extract_xml_answer(response) for response in responses]
    return [0.5 if response.isdigit() else 0.0 for response in extracted_responses]

# train: 7,473 examples, non-integer final answers: 0
# test: 1,319 examples, non-integer final answers: 0

In [13]:
def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward the exact newline-separated XML reasoning/answer structure."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = completion_texts(completions)
    matches = [re.match(pattern, response, flags=re.DOTALL) for response in responses]
    return [0.5 if match else 0.0 for match in matches]


def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward having XML reasoning and answer tags in the correct order."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = completion_texts(completions)
    matches = [re.match(pattern, response, flags=re.DOTALL) for response in responses]
    return [0.5 if match else 0.0 for match in matches]

In [14]:
def count_xml(text) -> float:
    """Reward correct XML tags and penalize extra text after answer tags."""
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1]) * 0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1) * 0.001
    return count


def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    """Granular XML format reward with penalties for trailing content."""
    contents = completion_texts(completions)
    return [count_xml(content) for content in contents]


reward_funcs = [
    xmlcount_reward_func,
    soft_format_reward_func,
    strict_format_reward_func,
    int_reward_func,
    correctness_reward_func,
]

print("Reward functions ready:")
for reward_func in reward_funcs:
    print(f"  - {reward_func.__name__}")

Reward functions ready:
  - xmlcount_reward_func
  - soft_format_reward_func
  - strict_format_reward_func
  - int_reward_func
  - correctness_reward_func


## Model + tokenizer (bf16 LoRA)

In [15]:
def load_model_and_tokenizer():
    print(f"Loading model: {model_name}")
    print(f"Max sequence length: {max_seq_length}")

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        trust_remote_code=True,
        dtype=torch.bfloat16,
        # attn_implementation="flash_attention_2",
    )
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    gemma_stop_token_ids = [1, 106]  # <eos>, <end_of_turn>
    end_of_turn_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
    if end_of_turn_id in gemma_stop_token_ids:
        tokenizer.eos_token = "<end_of_turn>"

    lora_config = LoraConfig(
        r=lora_rank,
        lora_alpha=lora_rank,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        lora_dropout=0.1,
        task_type=TaskType.CAUSAL_LM,
    )
    print("Applying LoRA adaptation to model...")
    model = get_peft_model(model, lora_config)
    print("LoRA Training Parameters Summary:")
    model.print_trainable_parameters()

    return model, tokenizer

In [16]:
model, tokenizer = load_model_and_tokenizer()

Loading model: gemma-3-1b-it
Max sequence length: 1024


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Applying LoRA adaptation to model...
LoRA Training Parameters Summary:
trainable params: 26,091,520 || all params: 1,025,977,472 || trainable%: 2.5431


In [17]:
# def score_one_completion(prompt, response, true_answer):
#     """Run each reward function on one generated response."""
#     completion = [[{"role": "assistant", "content": response}]]
#     scores = {}

#     for reward_func in reward_funcs:
#         kwargs = {
#             "prompts": [prompt],
#             "completions": completion,
#             "answer": [true_answer],
#         }
#         score = reward_func(**kwargs)[0]
#         scores[reward_func.__name__] = float(score)

#     scores["total"] = sum(scores.values())
#     return scores


# @torch.no_grad()
# def generate_raw_response(prompt, max_new_tokens=256):
#     """Generate one answer from the current model before GRPO training."""
#     text = tokenizer.apply_chat_template(
#         prompt,
#         add_generation_prompt=True,
#         tokenize=False,
#     )
#     inputs = tokenizer(text, return_tensors="pt").to(model.device)

#     output_ids = model.generate(
#         **inputs,
#         max_new_tokens=max_new_tokens,
#         do_sample=True,
#         temperature=0.7,
#         top_p=0.95,
#         pad_token_id=tokenizer.pad_token_id,
#     )

#     new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
#     return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# def inspect_raw_model_rewards(num_prompts=10, max_new_tokens=256):
#     """Print raw model responses and reward breakdowns for a few training prompts."""
#     model.eval()

#     for index, example in enumerate(dataset.select(range(num_prompts)), start=1):
#         prompt = example["prompt"]
#         question = prompt[-1]["content"]
#         true_answer = example["answer"]
#         response = generate_raw_response(prompt, max_new_tokens=max_new_tokens)
#         scores = score_one_completion(prompt, response, true_answer)

#         print("=" * 100)
#         print(f"Example {index}")
#         print(f"Question: {question}")
#         print(f"True answer: {true_answer}")
#         print("\nRaw model response:")
#         print(response)
#         print("\nReward scores:")
#         for name, score in scores.items():
#             print(f"  {name:32s} {score:7.3f}")


# inspect_raw_model_rewards(num_prompts=10, max_new_tokens=256)

In [18]:
!nvidia-smi

Mon Jul  6 04:55:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:CE:00.0 Off |                    0 |
|  0%   46C    P0            111W /  300W |    2353MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## GRPO training configuration

In [19]:
def build_training_args():
    return GRPOConfig(
        learning_rate=5e-6,
        max_grad_norm=0.1,
        bf16=True,

        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_generations=8,
        max_completion_length=786,
        # max_prompt_length=max_prompt_length,
        # max_completion_length=max_seq_length - max_prompt_length, #768
        # mask_truncated_completions=True,

        num_train_epochs=1,
        save_steps=250,
        logging_steps=1,

        output_dir="./gemma3_1b_gsm8k_grpo_outputs",
        report_to="trackio",
        log_on_each_node=False,
    )

### GRPO config with vLLM rollout (colocate)

In [20]:
def build_training_args_vllm():
    return GRPOConfig(
        learning_rate=5e-6,
        max_grad_norm=0.1,
        bf16=True,

        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_generations=8,
        max_completion_length=768,
        # mask_truncated_completions=True,

        num_train_epochs=1,
        save_steps=250,
        logging_steps=1,

        use_vllm=True,
        vllm_mode="colocate",
        vllm_gpu_memory_utilization=0.3, 
        vllm_max_model_length=1280,
        # vllm_tensor_parallel_size=1,
        vllm_enable_sleep_mode=False,
        generation_kwargs={"stop_token_ids": [1, 106]},

        output_dir="./gemma3_1b_gsm8k_grpo_outputs",
        report_to="trackio",
        log_on_each_node=False,
    )

# generation_batch_size is auto calculated based on generation_batch_size = per_device_train_batch_size × num_processes × steps_per_generation
# vllm_importance_sampling_correction=True # default

training_args = build_training_args_vllm()
print("vLLM rollout enabled :", training_args.use_vllm)
print("vLLM mode            :", training_args.vllm_mode)
print("vLLM GPU mem fraction:", training_args.vllm_gpu_memory_utilization)
print("generation_batch_size:", training_args.generation_batch_size)

vLLM rollout enabled : True
vLLM mode            : colocate
vLLM GPU mem fraction: 0.3
generation_batch_size: 16


## Experiment tracking (trackio)

In [21]:
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
run_name = f"simple-reward-{timestamp}"
trackio.init(
    project="GRPO-Mathematical-Reasoning",
    name=run_name,
    config={
        "model_name": model_name,
        "dataset": "GSM8K",
        "technique": "GRPO + LoRA (bf16)",
        "learning_rate": training_args.learning_rate,
        "batch_size": training_args.per_device_train_batch_size,
        "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
        "effective_batch_size": (
            training_args.per_device_train_batch_size
            * training_args.gradient_accumulation_steps
        ),
        "max_steps": training_args.max_steps,
        "lora_r": lora_rank,
        "lora_alpha": lora_rank,
        "num_generations": training_args.num_generations,
        "max_prompt_length": max_prompt_length,
        "max_completion_length": training_args.max_completion_length,
        "num_reward_functions": 5,
        "reward_set": "hf_grpo_lesson_exact",
        "mask_truncated_completions": training_args.mask_truncated_completions,
    },
)
print(f"Run name: {run_name}")

* Trackio project initialized: GRPO-Mathematical-Reasoning
* Trackio metrics logged to: /root/.cache/huggingface/trackio
* NVIDIA GPU detected, enabling automatic GPU metrics logging
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: simple-reward-20260706-045643


Run name: simple-reward-20260706-045643


## Build GRPOTrainer and train

In [22]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_funcs,
    args=training_args,
    train_dataset=dataset,
)
print("GRPO Trainer initialized successfully.")
print(f"Training dataset: {len(dataset):,} examples")
print(f"Reward functions: {len(trainer.reward_funcs)} active")

/usr/local/lib/python3.11/dist-packages/trl/generation/vllm_generation.py:294: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.19.0. You have version 0.21.0 installed. We recommend installing a supported version to avoid compatibility issues.
  if not is_vllm_available():


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 7/7 [00:00<00:00, 27.03it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 5/5 [00:00<00:00, 40.61it/s]


GRPO Trainer initialized successfully.
Training dataset: 7,473 examples
Reward functions: 5 active


### Essential GRPO training metrics for Trackio

Reward-over-time is already logged by TRL as `reward`. Keep `reward`, `reward_std`, `frac_reward_zero_std`, and `entropy` on the Trackio dashboard.

This probe adds completion-quality metrics:

- `completions/mean_length`
- `completions/max_length`
- `completions/clipped_ratio`
- `format/exact_rate`
- `answers/extracted_rate`
- `answers/exact_rate`

In [23]:
def attach_essential_grpo_metrics(trainer, tokenizer):
    """Add simple Trackio metrics without changing GRPO training behavior."""
    import math
    import types

    def attach_reward_std_filter():
        if getattr(trainer, "_reward_func_std_filter_attached", False):
            return

        original_log = trainer.log

        def log_without_reward_func_std(self, logs, start_time=None):
            for split in ["train", "eval"]:
                metric_store = getattr(self, "_metrics", {}).get(split, {})
                for key in list(metric_store.keys()):
                    if key.startswith("rewards/") and key.endswith("/std"):
                        metric_store.pop(key, None)
            return original_log(logs, start_time)

        trainer.log = types.MethodType(log_without_reward_func_std, trainer)
        trainer._reward_func_std_filter_attached = True

    attach_reward_std_filter()

    if getattr(trainer, "_essential_grpo_metrics_attached", False):
        print("Essential GRPO metrics are already attached.")
        print("Reward-function std metrics are filtered from Trackio logs.")
        return trainer

    original_calculate_rewards = trainer._calculate_rewards

    def log_metric(store, name, value):
        if value is None:
            return
        if isinstance(value, torch.Tensor):
            value = value.detach().float().item()
        if isinstance(value, (int, float)) and math.isfinite(value):
            store[name].append(float(value))

    def completion_text(completion):
        if isinstance(completion, list) and completion and isinstance(completion[0], dict):
            return completion[0].get("content", "")
        return str(completion)

    def clean_number(text):
        if text is None:
            return None
        text = str(text).strip().replace(",", "")
        text = re.sub(r"\s+", "", text)
        return text or None

    def _calculate_rewards_with_metrics(self, inputs, prompts, completions, completion_ids_list):
        rewards_per_func = original_calculate_rewards(inputs, prompts, completions, completion_ids_list)
        store = self._pending_metrics

        lengths = [len(ids) for ids in completion_ids_list]
        if lengths:
            log_metric(store, "completions/mean_length", sum(lengths) / len(lengths))
            log_metric(store, "completions/max_length", max(lengths))

            max_len = getattr(self.args, "max_completion_length", None)
            if max_len is not None:
                eos_or_pad = {0, 1, 106}  # <pad>, <eos>, <end_of_turn>
                clipped = []
                for ids in completion_ids_list:
                    hit_length_limit = len(ids) >= max_len
                    ended_cleanly = bool(ids) and ids[-1] in eos_or_pad
                    clipped.append(float(hit_length_limit and not ended_cleanly))
                log_metric(store, "completions/gemma_clipped_ratio", sum(clipped) / len(clipped))

        responses = [completion_text(completion) for completion in completions]
        if responses:
            exact_format = []
            extracted_answers = []
            exact_answers = []

            for response, example in zip(responses, inputs):
                format_match = match_format.search(response)
                exact_format.append(format_match is not None)

                predicted = format_match.group(1) if format_match else None
                predicted = clean_number(predicted)
                expected = clean_number(example.get("answer"))

                extracted_answers.append(predicted is not None)
                exact_answers.append(predicted is not None and predicted == expected)

            log_metric(store, "format/exact_rate", sum(exact_format) / len(exact_format))
            log_metric(store, "answers/extracted_rate", sum(extracted_answers) / len(extracted_answers))
            log_metric(store, "answers/exact_rate", sum(exact_answers) / len(exact_answers))

        return rewards_per_func

    trainer._calculate_rewards = types.MethodType(_calculate_rewards_with_metrics, trainer)
    trainer._essential_grpo_metrics_attached = True

    print("Attached essential GRPO metrics for Trackio.")
    print("Filtered per-reward-function std metrics; kept overall reward_std.")
    print("Dashboard basics: reward, reward_std, frac_reward_zero_std, entropy, completions/*, format/*, answers/*.")
    return trainer

In [24]:
trainer = attach_essential_grpo_metrics(trainer, tokenizer)

Attached essential GRPO metrics for Trackio.
Filtered per-reward-function std metrics; kept overall reward_std.
Dashboard basics: reward, reward_std, frac_reward_zero_std, entropy, completions/*, format/*, answers/*.


### Held-out eval during GRPO training

In [25]:
from transformers.trainer_utils import IntervalStrategy


def load_gsm8k_eval_dataset(max_examples=128, seed=3407):
    """Load a small held-out GSM8K test subset for fast eval during RL."""
    eval_data = load_dataset(
        "parquet",
        data_files={"test": str(gsm8k_local_dir / "test-00000-of-00001.parquet")},
        split="test",
    )
    eval_data = eval_data.map(process_dataset_example)

    if max_examples is not None and len(eval_data) > max_examples:
        eval_data = eval_data.shuffle(seed=seed).select(range(max_examples))

    print(f"Held-out eval examples: {len(eval_data):,}")
    print(f"Sample eval answer: {eval_data[0]['answer']}")
    return eval_data


def add_eval_steps_to_trainer(
    trainer,
    eval_dataset,
    eval_steps=50,
    num_generations_eval=4,
    eval_on_start=True,
):
    """Configure scheduled held-out eval without rebuilding the trainer."""
    trainer.eval_dataset = eval_dataset

    trainer.args.eval_strategy = IntervalStrategy.STEPS
    trainer.args.eval_steps = eval_steps
    trainer.args.eval_on_start = eval_on_start
    trainer.args.per_device_eval_batch_size = trainer.args.per_device_train_batch_size

    trainer.args.num_generations_eval = num_generations_eval
    trainer.num_generations_eval = num_generations_eval

    print("Held-out eval enabled.")
    print(f"  eval_strategy        : {trainer.args.eval_strategy}")
    print(f"  eval_steps           : {trainer.args.eval_steps}")
    print(f"  eval_on_start        : {trainer.args.eval_on_start}")
    print(f"  num_generations_eval : {trainer.num_generations_eval}")
    print("Trackio eval metrics will use the eval_ prefix, for example eval_reward and eval_answers/exact_rate.")
    return trainer


eval_dataset = load_gsm8k_eval_dataset(max_examples=128)
trainer = add_eval_steps_to_trainer(
    trainer,
    eval_dataset=eval_dataset,
    eval_steps=250,
    num_generations_eval=4,
    eval_on_start=True,
)

Held-out eval examples: 128
Sample eval answer: 160
Held-out eval enabled.
  eval_strategy        : IntervalStrategy.STEPS
  eval_steps           : 250
  eval_on_start        : True
  num_generations_eval : 4
Trackio eval metrics will use the eval_ prefix, for example eval_reward and eval_answers/exact_rate.


In [26]:
def prepare_grpo_debug_batch(trainer):
    """Build one real GRPO batch without taking an optimizer step."""
    trainer.model.train()

    raw_batch = next(iter(trainer.get_train_dataloader()))
    batch = trainer._prepare_inputs(raw_batch)

    prompt_ids = batch["prompt_ids"]
    completion_ids = batch["completion_ids"]
    completion_mask = batch["completion_mask"].bool()
    prompt_completion_ids = torch.cat([prompt_ids, completion_ids], dim=1)
    attention_mask = torch.cat([batch["prompt_mask"], batch["completion_mask"]], dim=1)
    logits_to_keep = completion_ids.size(1)

    with torch.no_grad():
        current_logps, entropies = trainer._get_per_token_logps_and_entropies(
            trainer.model,
            prompt_completion_ids,
            attention_mask,
            logits_to_keep,
            batch_size=completion_ids.size(0),
            compute_entropy=True,
            pixel_values=batch.get("pixel_values"),
            image_grid_thw=batch.get("image_grid_thw"),
            num_images=batch.get("num_images"),
            pixel_attention_mask=batch.get("pixel_attention_mask"),
            image_sizes=batch.get("image_sizes"),
            token_type_ids=batch.get("token_type_ids"),
            mm_token_type_ids=batch.get("mm_token_type_ids"),
            image_position_ids=batch.get("image_position_ids"),
        )

    debug = {
        "batch": batch,
        "completion_ids": completion_ids,
        "completion_mask": completion_mask,
        "current_logps": current_logps,
        "entropies": entropies,
    }

    print("Prepared one real GRPO debug batch.")
    print(f"completions shape: {tuple(completion_ids.shape)}")
    print(f"valid completion tokens: {int(completion_mask.sum().item())}")
    print(f"has vLLM logprobs: {batch.get('sampling_per_token_logps') is not None}")
    print(f"has TRL old logprobs: {batch.get('old_per_token_logps') is not None}")
    print(f"has importance sampling ratio: {batch.get('importance_sampling_ratio') is not None}")
    return debug


grpo_debug = prepare_grpo_debug_batch(trainer)


Prepared one real GRPO debug batch.
completions shape: (8, 768)
valid completion tokens: 2457
has vLLM logprobs: True
has TRL old logprobs: True
has importance sampling ratio: True


In [27]:
def print_vllm_vs_trl_logprobs(debug, tokenizer, num_completions_to_show=2, num_tokens_to_show=24):
    """Print token-level vLLM sampler logprobs beside TRL training logprobs."""
    batch = debug["batch"]
    completion_ids = debug["completion_ids"]
    completion_mask = debug["completion_mask"]
    current_logps = debug["current_logps"]

    vllm_logps = batch.get("sampling_per_token_logps")
    old_logps = batch.get("old_per_token_logps")

    if vllm_logps is None:
        print("No vLLM sampling_per_token_logps found.")
        print("Check use_vllm=True and vllm_importance_sampling_correction=True.")
        return

    rows_to_show = min(num_completions_to_show, completion_ids.size(0))
    for row in range(rows_to_show):
        valid_positions = torch.where(completion_mask[row])[0].tolist()[:num_tokens_to_show]
        decoded = tokenizer.decode(completion_ids[row][completion_mask[row]], skip_special_tokens=False)

        print("\n" + "=" * 100)
        print(f"Completion {row} | valid tokens: {int(completion_mask[row].sum().item())}")
        print("Decoded prefix:")
        print(decoded[:800])
        print("\nToken logprobs:")
        print(f"{'i':>3} {'token_id':>8} {'token':>18} {'vllm_logp':>12} {'old_trl':>12} {'current_trl':>12} {'old-vllm':>12} {'exp(old-vllm)':>14}")

        for pos in valid_positions:
            token_id = int(completion_ids[row, pos].item())
            token_text = tokenizer.decode([token_id], skip_special_tokens=False).replace("\n", "\\n")
            if len(token_text) > 16:
                token_text = token_text[:15] + "..."

            vllm_lp = float(vllm_logps[row, pos].item())
            old_lp = float(old_logps[row, pos].item()) if old_logps is not None else float("nan")
            cur_lp = float(current_logps[row, pos].item())
            diff = old_lp - vllm_lp
            ratio = float(torch.exp(torch.tensor(diff)).item())

            print(f"{pos:3d} {token_id:8d} {token_text!r:>18} {vllm_lp:12.4f} {old_lp:12.4f} {cur_lp:12.4f} {diff:12.4f} {ratio:14.4f}")

    mask = completion_mask
    if old_logps is not None:
        diff = (old_logps - vllm_logps)[mask]
        cur_old_diff = (current_logps - old_logps)[mask]
        print("\nSummary over valid completion tokens:")
        print(f"mean(old_trl - vllm):       {diff.mean().item():.6f}")
        print(f"mean(abs(old_trl - vllm)):  {diff.abs().mean().item():.6f}")
        print(f"max(abs(old_trl - vllm)):   {diff.abs().max().item():.6f}")
        print(f"mean(current - old_trl):    {cur_old_diff.mean().item():.6f}")
        print(f"mean(abs(current - old)):   {cur_old_diff.abs().mean().item():.6f}")


print_vllm_vs_trl_logprobs(
    grpo_debug,
    tokenizer,
    num_completions_to_show=2,
    num_tokens_to_show=24,
)



Completion 0 | valid tokens: 275
Decoded prefix:
<reasoning>
The question asks for the minimum grade Ahmed needs to achieve to beat Emily. We are given that the final assignment is worth the same as all the other assignments. We are given that Ahmed has a 91 and Emily has a 92. Emily got a 90 on the final assignment. We need to find the minimum grade Ahmed needs to get to beat Emily. This means we need to find a number that when added to Ahmed's current grade, results in a grade higher than Emily's. We can rephrase this as a comparison problem: Ahmed needs to be at least 1 grade higher than Emily.

<answer>
Let Ahmed's grade be 'a' and Emily's grade be 'e'. We are given that:
- Ahmed's grade: a = 91
- Emily's grade: e = 92
- Final assignment: The final assignment is worth the same as all other assignments.
- Emily's final grade: e = 90



Token logprobs:
  i token_id              token    vllm_logp      old_trl  current_trl     old-vllm  exp(old-vllm)
  0   236820                '<'  

In [28]:
def print_entropy_importance_and_clip(debug, trainer):
    """Print the main RL internals for the cached GRPO debug batch."""
    batch = debug["batch"]
    completion_mask = debug["completion_mask"]
    current_logps = debug["current_logps"]
    entropies = debug["entropies"]
    old_logps = batch.get("old_per_token_logps")
    importance_ratio = batch.get("importance_sampling_ratio")

    mask = completion_mask
    valid_count = mask.sum().clamp(min=1)

    print("Entropy over valid completion tokens:")
    if entropies is None:
        print("  Not available. Re-run the prepare cell with compute_entropy=True.")
    else:
        valid_entropy = entropies[mask]
        print(f"  mean: {valid_entropy.mean().item():.6f}")
        print(f"  min : {valid_entropy.min().item():.6f}")
        print(f"  max : {valid_entropy.max().item():.6f}")

    print("\nSampling/importance Sampling Ratio:")
    if importance_ratio is None:
        print("  Not available in this batch.")
    else:
        ratio = importance_ratio.detach().float()
        if ratio.ndim == 2 and ratio.shape == mask.shape:
            ratio_values = ratio[mask]
        else:
            ratio_values = ratio.reshape(-1)
        print(f"  min : {ratio_values.min().item():.6f}")
        print(f"  mean: {ratio_values.mean().item():.6f}")
        print(f"  max : {ratio_values.max().item():.6f}")

    print("\nClip Ratio:")
    if old_logps is None:
        print("  Not available because old_per_token_logps is missing.")
        return

    log_ratio = current_logps - old_logps
    policy_ratio = torch.exp(log_ratio)

    advantages = batch["advantages"]
    if advantages.dim() == 1:
        advantages = advantages.unsqueeze(1)

    epsilon = getattr(trainer, "epsilon", 0.2)
    epsilon_low = getattr(trainer, "epsilon_low", epsilon)
    epsilon_high = getattr(trainer, "epsilon_high", epsilon)

    low_clipped = (policy_ratio < 1 - epsilon_low) & (advantages < 0) & mask
    high_clipped = (policy_ratio > 1 + epsilon_high) & (advantages > 0) & mask
    region_clipped = low_clipped | high_clipped

    valid_policy_ratio = policy_ratio[mask]
    print(f"  policy ratio min : {valid_policy_ratio.min().item():.6f}")
    print(f"  policy ratio mean: {valid_policy_ratio.mean().item():.6f}")
    print(f"  policy ratio max : {valid_policy_ratio.max().item():.6f}")
    print(f"  low clipped      : {(low_clipped.float().sum() / valid_count).item():.6f}")
    print(f"  high clipped     : {(high_clipped.float().sum() / valid_count).item():.6f}")
    print(f"  region clipped   : {(region_clipped.float().sum() / valid_count).item():.6f}")


print_entropy_importance_and_clip(grpo_debug, trainer)


Entropy over valid completion tokens:
  mean: 0.361328
  min : 0.000000
  max : 3.000000

Sampling/importance Sampling Ratio:
  min : 0.253718
  mean: 0.861164
  max : 2.170028

Clip Ratio:
  policy ratio min : 1.000000
  policy ratio mean: 1.000000
  policy ratio max : 1.000000
  low clipped      : 0.000000
  high clipped     : 0.000000
  region clipped   : 0.000000


In [29]:
# Debug run: keep training short while inspecting the RL internals.
# Change this to None for a full configured training run.
DEBUG_MAX_TRAIN_STEPS = 5

if DEBUG_MAX_TRAIN_STEPS is not None:
    trainer.args.max_steps = DEBUG_MAX_TRAIN_STEPS
    trainer.args.eval_on_start = False
    trainer.args.eval_steps = DEBUG_MAX_TRAIN_STEPS
    trainer.args.save_steps = DEBUG_MAX_TRAIN_STEPS
    trainer.args.logging_steps = 1
    print(f"Debug training run enabled: trainer.train() will stop after {DEBUG_MAX_TRAIN_STEPS} steps.")
else:
    print("Full training run enabled.")


Debug training run enabled: trainer.train() will stop after 5 steps.


In [30]:
print("Starting GRPO training...")
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 106}.


Starting GRPO training...


* Run finished. Uploading logs to Trackio (please wait...)
* Trackio project initialized: huggingface
* Trackio metrics logged to: /root/.cache/huggingface/trackio
* NVIDIA GPU detected, enabling automatic GPU metrics logging
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: calm-river-2


Step,Training Loss,Validation Loss,Num Tokens,Completions/mean Length,Completions/min Length,Completions/max Length,Completions/clipped Ratio,Completions/mean Terminated Length,Completions/min Terminated Length,Completions/max Terminated Length,Rewards/xmlcount Reward Func/mean,Rewards/soft Format Reward Func/mean,Rewards/strict Format Reward Func/mean,Rewards/int Reward Func/mean,Rewards/correctness Reward Func/mean,Reward,Reward Std,Frac Reward Zero Std,Answers/exact Rate,Answers/extracted Rate,Completions/gemma Clipped Ratio,Format/exact Rate,Sampling/sampling Logp Difference/mean,Sampling/sampling Logp Difference/max,Sampling/importance Sampling Ratio/min,Sampling/importance Sampling Ratio/mean,Sampling/importance Sampling Ratio/max,Entropy,Clip Ratio/low Mean,Clip Ratio/low Min,Clip Ratio/high Mean,Clip Ratio/high Max,Clip Ratio/region Mean
5,0.191703,0.021844,27299.000000,235.806641,109.484375,490.484375,0.085938,188.672566,109.484375,323.156250,-0.189180,0.200195,0.000000,0.168945,0.332031,0.511992,0.983527,0.000000,0.105469,0.400391,0.085938,0.400391,0.013402,1.040720,0.224819,0.836961,1.788167,0.179847,0.000000,0.000000,0.000000,0.000000,0.000000


* Run finished. Uploading logs to Trackio (please wait...)


TrainOutput(global_step=5, training_loss=0.046611569821834564, metrics={'train_runtime': 1001.7472, 'train_samples_per_second': 0.08, 'train_steps_per_second': 0.005, 'total_flos': 0.0, 'train_loss': 0.046611569821834564, 'epoch': 0.0013383297644539614})

In [ ]:
try:
    trackio.finish()
except RuntimeError as exc:
    print(f"Trackio finish skipped: {exc}")

print("Training completed successfully.")
print(f"Model saved to: {training_args.output_dir}")

## Save model

In [ ]:
model.save_pretrained("grpo_saved_lora")
tokenizer.save_pretrained("grpo_saved_lora")

## Qualitative evaluation

In [ ]:
def test_model(model, tokenizer, question, max_length=512):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    print(f"Processing: {question}")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
            length_penalty=1.0,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_text = response[len(text) :].strip()
    return generated_text

In [ ]:
gsm8k_question = (
    "Natalia sold clips to 48 of her friends in April, and then she sold "
    "half as many clips in May. How many clips did Natalia sell altogether "
    "in April and May?"
)
expected_answer = "72"
gsm8k_response = test_model(model, tokenizer, gsm8k_question, max_length=768)

print(f"Question: {gsm8k_question}")
print(f"Model Response:\n{gsm8k_response}")

has_reasoning = reasoning_start in gsm8k_response and reasoning_end in gsm8k_response
has_solution = solution_start in gsm8k_response and solution_end in gsm8k_response
print("\nFormat Check:")
print(f"Reasoning section: {has_reasoning}")
print(f"Solution section: {has_solution}")
if has_solution:
    try:
        solution_text = (
            gsm8k_response.split(solution_start)[1]
            .split(solution_end)[0]
            .strip()
        )
        extracted_number = "".join(filter(str.isdigit, solution_text))
        expected_number = "".join(filter(str.isdigit, expected_answer))
        is_correct = extracted_number == expected_number
        print(f"Extracted: {solution_text}")
        print(f"Expected: {expected_answer}")
        print(f"Correct: {is_correct}")
    except IndexError:
        print("Could not extract solution")